In [1]:
# Set project paths.
from pathlib import Path
import os
import sys

def find_project_root():
    current = Path.cwd()

    for folder in [current] + list(current.parents):
        if (folder / "Data").exists() and (folder / "Notebooks").exists():
            return folder

    raise FileNotFoundError("Could not find project root. Make sure Data and Notebooks folders exist.")

project_folder = find_project_root()
notebook_folder = project_folder / "Notebooks"

os.chdir(project_folder)

print("Project folder:", project_folder)
print("Notebook folder:", notebook_folder)

Project folder: /Users/mac/Library/CloudStorage/OneDrive-UniversityofKeele/Dissertation/send-ev-project
Notebook folder: /Users/mac/Library/CloudStorage/OneDrive-UniversityofKeele/Dissertation/send-ev-project/Notebooks


In [2]:
# Import packages and helpers.
os.chdir(notebook_folder)

import import_ipynb
import pandas as pd
import plotly.express as px

from sklearn.metrics import r2_score, mean_absolute_error

from Data_Loader import DataLoader

os.chdir(notebook_folder)

from Training_Models import ConsumptionModel

os.chdir(project_folder)

print("Current folder:", os.getcwd())

Project folder: /Users/mac/Library/CloudStorage/OneDrive-UniversityofKeele/Dissertation/send-ev-project
Notebook folder: /Users/mac/Library/CloudStorage/OneDrive-UniversityofKeele/Dissertation/send-ev-project/Notebooks
                     air_temp  gti  surface_pressure  snow_depth  \
DateTime                                                           
2023-01-01 00:05:00         9    0             978.0         0.0   
2023-01-01 00:10:00         9    0             978.1         0.0   
2023-01-01 00:15:00         9    0             978.2         0.0   
2023-01-01 00:20:00         9    0             978.2         0.0   
2023-01-01 00:25:00         9    0             978.3         0.0   

                     cloud_opacity  ghi  clearsky_gti  wind_speed_100m  \
DateTime                                                                 
2023-01-01 00:05:00           33.8    0             0             11.0   
2023-01-01 00:10:00           26.3    0             0             11.2   
2023-01-

In [3]:
# Load data.
solcast, deop, expected = DataLoader.load_training_data()
deop_2022, solcast_2022, expected_2022 = DataLoader.load_testing_data()

print(deop.head())
print(solcast.head())
print(deop_2022.head())

                     power-con-ave  power-gen-wt-ave  power-gen-pv-ave
DateTime                                                              
2023-01-01 00:05:00       1317.487           573.029               0.0
2023-01-01 00:10:00       1319.056           612.280               0.0
2023-01-01 00:15:00       1338.842           652.990               0.0
2023-01-01 00:20:00       1329.313           813.526               0.0
2023-01-01 00:25:00       1338.366           643.773               0.0
                     air_temp  gti  surface_pressure  snow_depth  \
DateTime                                                           
2023-01-01 00:05:00         9    0             978.0         0.0   
2023-01-01 00:10:00         9    0             978.1         0.0   
2023-01-01 00:15:00         9    0             978.2         0.0   
2023-01-01 00:20:00         9    0             978.2         0.0   
2023-01-01 00:25:00         9    0             978.3         0.0   

                     cloud

In [4]:
# Choose consumption features.
con_features = [
    "air_temp", "heating_demand", "cooling_demand",
    "hour", "day_of_week", "month", "is_holiday",
    "non_working_day", "non_working_hour_interaction",
    "is_term_time",
    "hour_sin", "hour_cos", "day_sin", "day_cos",
    "smart_lag", "load_lag_7d",
    "yesterday_daily_mean", "yesterday_daily_max",
    "rolling_7d_baseline",
]

In [5]:
# Train consumption model.
con = ConsumptionModel(features=con_features)

con_preds, con_model = con.train_and_test(
    deop,
    solcast,
    deop_2022,
    solcast_2022,
    days_ahead=1,
)

Training new consumption model
5-Minute R2 Score: 0.8826 | MAE: 152.80 kW | MBE: -24.21 kW
Hourly R2 Score:   0.8965 | MAE: 140.92 kW
Daily R2 Score:    0.9315 | MAE: 92.71 kW


In [6]:
# Build results table.
consumption_results = pd.DataFrame({
    "actual": deop_2022.loc[con_preds.index, "power-con-ave"],
    "predicted": con_preds,
})

consumption_results.head()

,actual,predicted
DateTime,,
2022-03-08 00:00:00,0.0,0.0
2022-03-08 00:05:00,0.0,0.0
2022-03-08 00:10:00,0.0,0.0
2022-03-08 00:15:00,0.0,0.0
2022-03-08 00:20:00,0.0,0.0


In [7]:
# Plot actual vs predicted.
hourly_results = consumption_results.resample("1h").mean()

fig = px.line(
    hourly_results,
    y=["actual", "predicted"],
    title="Consumption: Actual vs Predicted",
    labels={"value": "Power [kW]", "DateTime": "Time"},
)

fig.show()

In [8]:
# Calculate monthly scores.
monthly_scores = []

for month, group in consumption_results.groupby(consumption_results.index.month):
    r2 = r2_score(group["actual"], group["predicted"])
    mae = mean_absolute_error(group["actual"], group["predicted"])

    monthly_scores.append({
        "month": month,
        "r2": r2,
        "mae": mae,
    })

monthly_scores = pd.DataFrame(monthly_scores)
monthly_scores

,month,r2,mae
0,3,0.000000,111.156877
1,4,0.723292,250.884345
2,5,0.819100,149.718676
3,6,0.699602,173.667376
4,7,0.866111,98.273352
5,8,0.814993,133.016275
6,9,0.792440,146.714084
7,10,0.768677,166.393611
8,11,0.892501,122.338775
9,12,0.673523,169.132829


In [9]:
# Plot monthly scores.
fig = px.bar(
    monthly_scores,
    x="month",
    y="r2",
    title="Consumption Model Monthly R2 Score",
    labels={"month": "Month", "r2": "R2 Score"},
)

fig.show()

In [10]:
# Run forecast horizons.
con_predictions = {}

for i in range(1, 15):
    print("Running", i, "day forecast")

    preds, _ = con.train_and_test(
        deop,
        solcast,
        deop_2022,
        solcast_2022,
        days_ahead=i,
    )

    con_predictions[f"{i}-Day Forecast"] = preds

Running 1 day forecast
Loading consumption model from Models/con_model_feats_2f482541.json
5-Minute R2 Score: 0.8826 | MAE: 152.80 kW | MBE: -24.21 kW
Hourly R2 Score:   0.8965 | MAE: 140.92 kW
Daily R2 Score:    0.9315 | MAE: 92.71 kW
Running 2 day forecast
Loading consumption model from Models/con_model_feats_2f482541.json
5-Minute R2 Score: 0.8624 | MAE: 163.69 kW | MBE: -17.90 kW
Hourly R2 Score:   0.8762 | MAE: 152.52 kW
Daily R2 Score:    0.9050 | MAE: 106.97 kW
Running 3 day forecast
Loading consumption model from Models/con_model_feats_2f482541.json
5-Minute R2 Score: 0.8414 | MAE: 174.64 kW | MBE: -13.33 kW
Hourly R2 Score:   0.8551 | MAE: 163.91 kW
Daily R2 Score:    0.8752 | MAE: 119.60 kW
Running 4 day forecast
Loading consumption model from Models/con_model_feats_2f482541.json
5-Minute R2 Score: 0.8282 | MAE: 180.49 kW | MBE: -7.61 kW
Hourly R2 Score:   0.8419 | MAE: 170.00 kW
Daily R2 Score:    0.8555 | MAE: 125.98 kW
Running 5 day forecast
Loading consumption model from 

In [11]:
# Compare horizons.
forecast_scores = []

for name, preds in con_predictions.items():
    actual = deop_2022.loc[preds.index, "power-con-ave"]

    r2 = r2_score(actual, preds)
    mae = mean_absolute_error(actual, preds)

    forecast_scores.append({
        "forecast": name,
        "r2": r2,
        "mae": mae,
    })

forecast_scores = pd.DataFrame(forecast_scores)
forecast_scores

,forecast,r2,mae
0,1-Day Forecast,0.882619,152.804279
1,2-Day Forecast,0.862391,163.691902
2,3-Day Forecast,0.841419,174.637957
3,4-Day Forecast,0.828238,180.490175
4,5-Day Forecast,0.817009,184.664626
5,6-Day Forecast,0.801185,190.855143
6,7-Day Forecast,0.784353,196.447306
7,8-Day Forecast,0.762459,203.541874
8,9-Day Forecast,0.734574,215.015933
9,10-Day Forecast,0.704982,225.614677


In [12]:
# Plot feature importance.
importance_values = con_model.feature_importances_

feature_importance = pd.Series(
    importance_values,
    index=con_features,
).sort_values()

fig = px.bar(
    x=feature_importance.values,
    y=feature_importance.index,
    orientation="h",
    title="Consumption Model Feature Importance",
    labels={"x": "Importance", "y": "Feature"},
)

fig.show()